In [9]:
print('ritu')

ritu


In [10]:
import ast
from datetime import datetime
from langchain_ai21.chat_models import ChatAI21
from langchain_groq import ChatGroq
from langchain_core.output_parsers import PydanticOutputParser
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_core.messages import SystemMessage,BaseMessage,HumanMessage,ToolMessage
from langgraph.types import interrupt,Command
from langgraph.graph import StateGraph,START,END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode
from typing import TypedDict,Annotated, List, Dict, Literal,Optional
from langgraph.checkpoint.postgres import PostgresSaver
from langsmith import traceable
from psycopg_pool import ConnectionPool
from pydantic import BaseModel
from json_repair import repair_json
import re
from typing import List
from dotenv import load_dotenv
import json
import os
load_dotenv()
DB_URL = os.getenv("PPT_URL")

class PptState(TypedDict):
    messages: Annotated[List[BaseMessage], add_messages]
    outline: List
    detailed_slides: List[Dict]
    current_slide_index: int
    feedback: str
    topic: str
    research_data: str
    action: Literal[ "continue_slide", "update_outline", "update_slide", "complete", '' ]
    tool_caller: Literal["generate_outline","generate_slide_detail"]
class DetailedPoint(BaseModel):
    key_point: str
    explanation: str
class DetailedSlideOutput(BaseModel):
    slide_number: int
    slide_title: str
    layout: Literal[
        "bullets",
        "bullets_with_text",
        "paragraph",
        "two_column",
        "mixed"
    ]
    intro_line: Optional[str] = None
    bullet_points: Optional[List[str]] = None
    supporting_text: Optional[str] = None
    paragraphs: Optional[List[str]] = None

model = ChatGroq(model="llama-3.3-70b-versatile")
# model = ChatAI21(model="jamba-mini-2-2026-01")
searchTool = TavilySearchResults(max_results=2)
tools = [searchTool]
model_with_tools = model.bind_tools(tools)
detailed_parser = PydanticOutputParser(pydantic_object= DetailedSlideOutput)

OUTLINE_SYSTEM_PROMPT = SystemMessage(
    content=f"""You are an expert presentation designer.

Your task:
Generate ONLY the presentation slide titles.

Example:
["Title 1", "Title 2", "Title 3"]
    
Strict Rules:
- Generate EXACTLY the number of slides requested by the user.
- Ensure logical flow from introduction to conclusion.
- Keep slide titles concise but descriptive.
- Use tools only if factual accuracy is required.
- Return a Python list of strings.
- No explanations.
""")
DETAIL_SYSTEM_PROMPT = SystemMessage(content=f"""
You are a professional presentation designer.

Your job is to generate visually balanced slide content for a PowerPoint presentation.

For each slide choose the most appropriate layout or a mix of layouts  and generate structured content.

Available layouts:
- bullets
- bullets_with_text
- paragraph
- mixed


{detailed_parser.get_format_instructions()}


Rules:
- Slides should be informative but not crowded.
- Include at least 1 statistic with source
- Use clear, short content (120-150 words)
- Bullet points: 5–7 items, short phrases
- Keep slide visually balanced
Return JSON only.
""")


@traceable(name="Generate Outline")
def generate_outline_node(state: PptState):
    """Step 1: Generate presentation outline"""
    messages = state["messages"] + [OUTLINE_SYSTEM_PROMPT]

    result = model_with_tools.invoke(messages)
    usage = result.response_metadata.get("token_usage", {})

    output = {
        'messages':[result],
        'current_slide_index':0,
        "tool_caller": "generate_outline",
        "usage": usage
            } 
    if result.content:
        try:
            print('result.content',result.content)
            output['outline'] = json.loads(result.content)
            try:
                print('ast.literal_eval',ast.literal_eval(result.content))
            except Exception as e:
                print(f'Exception outline',e)
        except json.JSONDecodeError as e:
            print('generate_outline_node',e)
    return output

@traceable(name="Generate Slide Detail")
def generate_slide_detail_node(state: PptState):
    """Generate detailed content using research data"""
    print('state',state['messages'])
    outline = state['outline']
    current_index = state['current_slide_index']
    detailed_slides = state.get('detailed_slides', [])
    total_slides = len(outline)
    current_slide = outline[current_index]
    
    
    if current_index >= total_slides:
        return {"action": "complete"}
    research = state.get('research_data', {})
    output = {
        "tool_caller": "generate_slide_detail",
        "current_slide_index": current_index
    }
    
    

    
    if state['action'] == "update_slide":
        feedback = state['feedback']
        last_slide = detailed_slides.pop()
        prompt = HumanMessage(content=f"""
Update this slide with research data:
Research: {research}
Current Content: {last_slide}
Feedback: {feedback}
        """)
        messages = [DETAIL_SYSTEM_PROMPT, prompt]  
        output['messages'] = messages



    else:

        prompt = HumanMessage(content=f"""
Slide: {current_slide}
Topic: {state['topic']}

If this slide needs current statistics, market data, or recent research, use the search tool first and get data till current {datetime.now().year} year.
For conceptual, instructional, or introductory slides, generate content directly without searching.
""")
        messages = [DETAIL_SYSTEM_PROMPT, prompt]
        output['messages'] = messages


        last_message = state["messages"][-1] if state["messages"] else None
        print('last_message',last_message)
        resuming_from_tool = (
            last_message is not None and 
            last_message.type == "tool"  # ToolMessage means search just completed
        )

        print('resuming_from_tool',resuming_from_tool)

        if resuming_from_tool:
            # Pass full history so model sees its own search results and can now generate
            messages = messages + state["messages"][-2:] 
        print('messages',messages)
        
    # print("\n\n\n\nmessages",messages)

    result = model_with_tools.invoke(messages)
    # print("\n\n\n\nresult",result)

    # usage = result.response_metadata.get("token_usage", {})
    # output["usage"] = usage
    
    if result.content:
        try:
            new_slide = detailed_parser.parse(repair_json(str(result.content))).model_dump()
            detailed_slides.append(new_slide)
            output['detailed_slides'] = detailed_slides
            output['current_slide_index'] = current_index + 1
            
        except Exception as e:
            print('error',e)
    
    if output['current_slide_index'] == total_slides:
        output["action"] = "complete"
    # output["action"] = "complete"
    print("output['messages']",output['messages'])
    output['messages'].append(result)
    print('after append',output['messages'])
    return output



def route_after_tools(state: PptState):
    return state["tool_caller"]

def human_decision(state: PptState):
    decision = interrupt({})
    if decision['action'] == "update_outline":

        return {
            'action': "update_outline",
            "messages":[decision['feedback']]
            }
    elif decision['action'] == 'continue_slide':
        return {'action':'continue_slide'}
    elif decision['action'] == 'update_slide':
        return {'action':'update_slide'}
def route_after_human(state: PptState):
    action = state['action']
    if action == 'update_outline':
        return "generate_outline"
    elif action in ('continue_slide', 'update_slide'):
        return "generate_slide_detail"
    elif action == 'complete':  
        return END
    return END

def tools_condition(state: PptState):
    """Route to tools if last message has tool calls"""
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    return "__end__"

def build_workflow():
    workflow = StateGraph(PptState)
    workflow.add_node("generate_outline", generate_outline_node)
    # workflow.add_node("research_slide", research_slide_node)
    workflow.add_node("generate_slide_detail", generate_slide_detail_node)
    workflow.add_node("human_decision", human_decision)
    workflow.add_node("tools", ToolNode(tools))
    
    workflow.add_edge(START, "generate_outline")

    workflow.add_conditional_edges( #done
        "generate_outline",
        tools_condition,
        {
            "tools": "tools",
            "__end__": "human_decision",
        },
    )

    workflow.add_conditional_edges(
        "human_decision",
        route_after_human,
        {
            "generate_outline": "generate_outline",
            "generate_slide_detail": "generate_slide_detail",
            END: END,
        },
    )
    
    

    workflow.add_conditional_edges(
        "generate_slide_detail",
        tools_condition,
        {
            "tools": "tools",
            "__end__": "human_decision",
        },
    )
    workflow.add_conditional_edges(
        "tools",
        route_after_tools,
        {
            "generate_outline": "generate_outline",
            "generate_slide_detail": "generate_slide_detail",
        },
    )

    return workflow
def create_ckeckpointer_and_graph(db_url: str):
    if not db_url:
        raise ValueError('Database Url environment variable not set')
    connection_kwargs = {
            "autocommit": True,
            "prepare_threshold": 0,
        }
    pool = ConnectionPool(
        conninfo=db_url,
            max_size=20,
            kwargs=connection_kwargs,
    )
    checkpointer = PostgresSaver(pool)
    checkpointer.setup()
    workflow = build_workflow()
    graph = workflow.compile(checkpointer=checkpointer)
    return checkpointer, graph

In [11]:
import random

topics = ["Impact of Social Media on Human Relationships",
"Plastic Pollution and Its Effect on the Environment",
"Importance of Mental Health in Modern Life",
"Urbanization and Its Impact on Cities",
"Healthy Lifestyle and the Role of Exercise",
"Electric Vehicles and the Future of Transportation",
"The Importance of Time Management for Students",
"Global Warming: Causes and Solutions",
"Women Empowerment in Modern Society",
"The Role of Education in Personal Development"]


checkpointer, graph = create_ckeckpointer_and_graph(DB_URL)

# topic = "Importance of Mental Health in Modern Life"
# topic = "Electric Vehicles and the Future of Transportation"

topic = random.choice(topics)
num_slides = 5
prompt = HumanMessage(
    content=f"""
Create EXACTLY {num_slides} slide titles for a presentation on:

Topic: {topic}

Do not create fewer or more slides.
"""
)
config = {'configurable':{'thread_id':'28-03-26-2'}}
state = {
            "messages": [prompt],
            "topic":topic,
            "outline": {},
            "detailed_slides": [],
            "current_slide_index": 0,
            "feedback": "",
            "action": "",
            "tool_caller": "generate_outline",
        }

result = graph.invoke(state,config = config)
print('run')


result.content ["Introduction to Social Media and Human Relationships", "The Rise of Social Media: Benefits and Drawbacks", "Social Media's Influence on Mental Health and Wellbeing", "The Effects of Social Media on Personal and Professional Relationships", "Conclusion: Navigating Healthy Social Media Habits"]
ast.literal_eval ['Introduction to Social Media and Human Relationships', 'The Rise of Social Media: Benefits and Drawbacks', "Social Media's Influence on Mental Health and Wellbeing", 'The Effects of Social Media on Personal and Professional Relationships', 'Conclusion: Navigating Healthy Social Media Habits']
run


In [19]:
state = Command(resume={
    "action":'continue_slide'
})
result2 = graph.invoke(state,config = config)

state [HumanMessage(content='\nCreate EXACTLY 5 slide titles for a presentation on:\n\nTopic: Impact of Social Media on Human Relationships\n\nDo not create fewer or more slides.\n', additional_kwargs={}, response_metadata={}, id='9f5952fd-bb90-4a7a-bd92-cb1ad30d98bd'), AIMessage(content='["Introduction to Social Media and Human Relationships", "The Rise of Social Media: Benefits and Drawbacks", "Social Media\'s Influence on Mental Health and Wellbeing", "The Effects of Social Media on Personal and Professional Relationships", "Conclusion: Navigating Healthy Social Media Habits"]', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 399, 'total_tokens': 457, 'completion_time': 0.2027151, 'completion_tokens_details': None, 'prompt_time': 0.02285832, 'prompt_tokens_details': None, 'queue_time': 0.06460462, 'total_time': 0.22557342}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_demand', '

In [17]:
result2

{'messages': [HumanMessage(content='\nCreate EXACTLY 5 slide titles for a presentation on:\n\nTopic: Impact of Social Media on Human Relationships\n\nDo not create fewer or more slides.\n', additional_kwargs={}, response_metadata={}, id='9f5952fd-bb90-4a7a-bd92-cb1ad30d98bd'),
  AIMessage(content='["Introduction to Social Media and Human Relationships", "The Rise of Social Media: Benefits and Drawbacks", "Social Media\'s Influence on Mental Health and Wellbeing", "The Effects of Social Media on Personal and Professional Relationships", "Conclusion: Navigating Healthy Social Media Habits"]', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 58, 'prompt_tokens': 399, 'total_tokens': 457, 'completion_time': 0.2027151, 'completion_tokens_details': None, 'prompt_time': 0.02285832, 'prompt_tokens_details': None, 'queue_time': 0.06460462, 'total_time': 0.22557342}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_68f543a7cc', 'service_tier': 'on_d

In [ ]:
state [HumanMessage(content='\nCreate EXACTLY 5 slide titles for a presentation on:\n\nTopic: Electric Vehicles and the Future of Transportation\n\nDo not create fewer or more slides.\n', additional_kwargs={}, response_metadata={}, id='79f9f33a-dab2-4c9e-b78f-6ef6b4dd20f5'), AIMessage(content='["Introduction to Electric Vehicles", "The Benefits of Electric Vehicles", "Challenges and Limitations of Electric Vehicles", "The Future of Transportation: Emerging Trends and Technologies", "Conclusion: Embracing a Sustainable Transportation Future"]', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 399, 'total_tokens': 445, 'completion_time': 0.154857142, 'completion_tokens_details': None, 'prompt_time': 0.046790351, 'prompt_tokens_details': None, 'queue_time': 0.161466158, 'total_time': 0.201647493}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d30ea-fe48-7403-939f-79f9b26d694c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 399, 'output_tokens': 46, 'total_tokens': 445})]
last_message content='["Introduction to Electric Vehicles", "The Benefits of Electric Vehicles", "Challenges and Limitations of Electric Vehicles", "The Future of Transportation: Emerging Trends and Technologies", "Conclusion: Embracing a Sustainable Transportation Future"]' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 399, 'total_tokens': 445, 'completion_time': 0.154857142, 'completion_tokens_details': None, 'prompt_time': 0.046790351, 'prompt_tokens_details': None, 'queue_time': 0.161466158, 'total_time': 0.201647493}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d30ea-fe48-7403-939f-79f9b26d694c-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 399, 'output_tokens': 46, 'total_tokens': 445}
resuming_from_tool False
messages [SystemMessage(content='\nYou are a professional presentation designer.\n\nYour job is to generate visually balanced slide content for a PowerPoint presentation.\n\nFor each slide choose the most appropriate layout or a mix of layouts  and generate structured content.\n\nAvailable layouts:\n- bullets\n- bullets_with_text\n- paragraph\n- mixed\n\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"slide_number": {"title": "Slide Number", "type": "integer"}, "slide_title": {"title": "Slide Title", "type": "string"}, "layout": {"enum": ["bullets", "bullets_with_text", "paragraph", "two_column", "mixed"], "title": "Layout", "type": "string"}, "intro_line": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Intro Line"}, "bullet_points": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Bullet Points"}, "supporting_text": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Supporting Text"}, "paragraphs": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Paragraphs"}}, "required": ["slide_number", "slide_title", "layout"]}\n```\n\n\nRules:\n- Slides should be informative but not crowded.\n- Include at least 1 statistic with source\n- Use clear, short content (120-150 words)\n- Bullet points: 5–7 items, short phrases\n- Keep slide visually balanced\nReturn JSON only.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\nSlide: Introduction to Electric Vehicles\nTopic: Electric Vehicles and the Future of Transportation\n\nIf this slide needs current statistics, market data, or recent research, use the search tool first and get data till current 2026 year.\nFor conceptual, instructional, or introductory slides, generate content directly without searching.\n', additional_kwargs={}, response_metadata={})]
output['messages'] [SystemMessage(content='\nYou are a professional presentation designer.\n\nYour job is to generate visually balanced slide content for a PowerPoint presentation.\n\nFor each slide choose the most appropriate layout or a mix of layouts  and generate structured content.\n\nAvailable layouts:\n- bullets\n- bullets_with_text\n- paragraph\n- mixed\n\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"slide_number": {"title": "Slide Number", "type": "integer"}, "slide_title": {"title": "Slide Title", "type": "string"}, "layout": {"enum": ["bullets", "bullets_with_text", "paragraph", "two_column", "mixed"], "title": "Layout", "type": "string"}, "intro_line": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Intro Line"}, "bullet_points": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Bullet Points"}, "supporting_text": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Supporting Text"}, "paragraphs": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Paragraphs"}}, "required": ["slide_number", "slide_title", "layout"]}\n```\n\n\nRules:\n- Slides should be informative but not crowded.\n- Include at least 1 statistic with source\n- Use clear, short content (120-150 words)\n- Bullet points: 5–7 items, short phrases\n- Keep slide visually balanced\nReturn JSON only.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\nSlide: Introduction to Electric Vehicles\nTopic: Electric Vehicles and the Future of Transportation\n\nIf this slide needs current statistics, market data, or recent research, use the search tool first and get data till current 2026 year.\nFor conceptual, instructional, or introductory slides, generate content directly without searching.\n', additional_kwargs={}, response_metadata={})]
after append [SystemMessage(content='\nYou are a professional presentation designer.\n\nYour job is to generate visually balanced slide content for a PowerPoint presentation.\n\nFor each slide choose the most appropriate layout or a mix of layouts  and generate structured content.\n\nAvailable layouts:\n- bullets\n- bullets_with_text\n- paragraph\n- mixed\n\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"slide_number": {"title": "Slide Number", "type": "integer"}, "slide_title": {"title": "Slide Title", "type": "string"}, "layout": {"enum": ["bullets", "bullets_with_text", "paragraph", "two_column", "mixed"], "title": "Layout", "type": "string"}, "intro_line": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Intro Line"}, "bullet_points": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Bullet Points"}, "supporting_text": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Supporting Text"}, "paragraphs": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Paragraphs"}}, "required": ["slide_number", "slide_title", "layout"]}\n```\n\n\nRules:\n- Slides should be informative but not crowded.\n- Include at least 1 statistic with source\n- Use clear, short content (120-150 words)\n- Bullet points: 5–7 items, short phrases\n- Keep slide visually balanced\nReturn JSON only.\n', additional_kwargs={}, response_metadata={}), HumanMessage(content='\nSlide: Introduction to Electric Vehicles\nTopic: Electric Vehicles and the Future of Transportation\n\nIf this slide needs current statistics, market data, or recent research, use the search tool first and get data till current 2026 year.\nFor conceptual, instructional, or introductory slides, generate content directly without searching.\n', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'm4rmqxbv8', 'function': {'arguments': '{"query":"electric vehicles market statistics 2026"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 814, 'total_tokens': 838, 'completion_time': 0.069229028, 'completion_tokens_details': None, 'prompt_time': 0.042044197, 'prompt_tokens_details': None, 'queue_time': 0.048020733, 'total_time': 0.111273225}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d30eb-0be9-73e0-b0e9-dd1ffdcd9196-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'electric vehicles market statistics 2026'}, 'id': 'm4rmqxbv8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 814, 'output_tokens': 24, 'total_tokens': 838})]
state [HumanMessage(content='\nCreate EXACTLY 5 slide titles for a presentation on:\n\nTopic: Electric Vehicles and the Future of Transportation\n\nDo not create fewer or more slides.\n', additional_kwargs={}, response_metadata={}, id='79f9f33a-dab2-4c9e-b78f-6ef6b4dd20f5'), AIMessage(content='["Introduction to Electric Vehicles", "The Benefits of Electric Vehicles", "Challenges and Limitations of Electric Vehicles", "The Future of Transportation: Emerging Trends and Technologies", "Conclusion: Embracing a Sustainable Transportation Future"]', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 399, 'total_tokens': 445, 'completion_time': 0.154857142, 'completion_tokens_details': None, 'prompt_time': 0.046790351, 'prompt_tokens_details': None, 'queue_time': 0.161466158, 'total_time': 0.201647493}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d30ea-fe48-7403-939f-79f9b26d694c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 399, 'output_tokens': 46, 'total_tokens': 445}), SystemMessage(content='\nYou are a professional presentation designer.\n\nYour job is to generate visually balanced slide content for a PowerPoint presentation.\n\nFor each slide choose the most appropriate layout or a mix of layouts  and generate structured content.\n\nAvailable layouts:\n- bullets\n- bullets_with_text\n- paragraph\n- mixed\n\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"slide_number": {"title": "Slide Number", "type": "integer"}, "slide_title": {"title": "Slide Title", "type": "string"}, "layout": {"enum": ["bullets", "bullets_with_text", "paragraph", "two_column", "mixed"], "title": "Layout", "type": "string"}, "intro_line": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Intro Line"}, "bullet_points": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Bullet Points"}, "supporting_text": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Supporting Text"}, "paragraphs": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Paragraphs"}}, "required": ["slide_number", "slide_title", "layout"]}\n```\n\n\nRules:\n- Slides should be informative but not crowded.\n- Include at least 1 statistic with source\n- Use clear, short content (120-150 words)\n- Bullet points: 5–7 items, short phrases\n- Keep slide visually balanced\nReturn JSON only.\n', additional_kwargs={}, response_metadata={}, id='535ddf52-7c91-43c7-b6c3-ee262b5b1b24'), HumanMessage(content='\nSlide: Introduction to Electric Vehicles\nTopic: Electric Vehicles and the Future of Transportation\n\nIf this slide needs current statistics, market data, or recent research, use the search tool first and get data till current 2026 year.\nFor conceptual, instructional, or introductory slides, generate content directly without searching.\n', additional_kwargs={}, response_metadata={}, id='e7d13a49-8a0f-4773-a726-673e057a681b'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'm4rmqxbv8', 'function': {'arguments': '{"query":"electric vehicles market statistics 2026"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 814, 'total_tokens': 838, 'completion_time': 0.069229028, 'completion_tokens_details': None, 'prompt_time': 0.042044197, 'prompt_tokens_details': None, 'queue_time': 0.048020733, 'total_time': 0.111273225}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d30eb-0be9-73e0-b0e9-dd1ffdcd9196-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'electric vehicles market statistics 2026'}, 'id': 'm4rmqxbv8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 814, 'output_tokens': 24, 'total_tokens': 838}), ToolMessage(content='[{"title": "Global Electric Car Sales and Electric Vehicle Statistics (2026)", "url": "https://tridenstechnology.com/electric-car-sales-statistics/", "content": "| Electric Car Model | Sales |\\n --- |\\n| Tesla Model Y | 357,582 |\\n| Tesla Model 3 | 192,440 |\\n| Chevy Equinox EV | 57,945 |\\n| Ford Mustang Mach-E | 51,620 |\\n| Hyundai Ioniq 5 | 47,039 |\\n| Honda Prologue | 41,194 |\\n| Ford F-150 Lightning | 27,307 |\\n| Rivian R1S | 24,852 |\\n| Chevy Blazer EV | 22,637 |\\n| Volkswagen ID.4 | 22,373 |\\n\\nConclusion\\n\\nBased on this research, we expect that the percentage of electric vehicles on our roads will increase to approximately 25% by the end of 2026.\\n\\nLoading ... Loading …\\n\\nIn conclusion, it seems that the future is looking bright for electric cars and 2026 is expected to be strong in EV sales.\\n\\nFor sure, we have just started the EV revolution!\\n\\nDo you agree? Drop the comment below. 😊\\n\\n## Ready to get started? [...] | Electric Car Model | Sales |\\n --- |\\n| Tesla Model Y | 8,360 |\\n| Tesla Model 3 | 3,335 |\\n| Volvo EX40 | 1,778 |\\n| Volvo EX30 | 1,683 |\\n| VW ID.7 | 1,599 |\\n| VW ID.4 | 1,594 |\\n| Skoda Elroq | 1,219 |\\n| Ford Explorer | 1,135 |\\n| Skoda Enyaq | 942 |\\n| Nissan Ariya | 897 |\\n\\n### France Electric Cars Sales\\n\\nIn Q1 2025, electric vehicle sales in France fell by 7% compared to the same period in 2024, however the full year sales were higher in 2025.\\n\\nRenault 5 was the best-selling model in Q4 2025, with 12,374 new registrations.\\n\\nThese are bestsellers in France: [...] In 2025, electric car sales continued to grow: ~473,348 fully electric cars were registered, representing around 23.4 % of all new cars sold across the year and marking a ~21.6 % increase over 2024.\\n\\nThese were UK’s best-selling manufacturers in 2025:\\n\\n| Brand | Sales |\\n --- |\\n| Ford | 8.5% |\\n| Tesla | 8.0% |\\n| BYD | 7.0% |\\n| VW | 6.6% |\\n| AUDI | 6.2% |\\n| Renault | 5.0% |\\n| BMW | 4.8% |\\n| Mercedes | 4.8% |\\n| Skoda | 4.1% |\\n| Hyundai | 4.0% |\\n\\nSource: CleanTechnica\\n\\n## Electric Car Sales in the US\\n\\nElectric car adoption in the US is happening at a faster pace than predicted.\\n\\nThe US charging infrastructure is getting built out fast, and Tesla’s opening the Supercharger network.\\n\\nElectric car sales in the USA reached new highs again in Q1 2025, continuing the long-term growth trend.", "score": 0.95838624}, {"title": "Electric Vehicle Forecast 2026: Global Growth, Policy Shifts, and Indu", "url": "https://ev-lectron.com/blogs/blog/electric-vehicle-forecast-2025-global-growth-policy-shifts-and-industry-outlook-leading-to-2026?srsltid=AfmBOorhtoH-Q7WMM8VwBmzv1P-rKQjMyoa-8aa1u_lNEk7G9Hej4aSt", "content": "## Global EV Market Outlook for 2026\\n\\nThe global EV market continues growing rapidly, but at a more measured pace than earlier years. Global sales of electric vehicles, including battery electric vehicles and plug-in hybrids, are expected to rise again in 2026 as electrification expands across passenger cars and light commercial vehicles.\\n\\nIn 2025, approximately 20.7 million EV units are expected to be registered globally, marking a 20% year-over-year growth. S&P Global Mobility projects global sales for battery electric passenger vehicles to post 15.1 million units for 2025, up by 30% compared to 2024 levels.\\n\\nA white Volvo EX30 electric SUV charging at an RACV public ultra-rapid charging station using a tethered CCS2 connector.", "score": 0.9338243}]', name='tavily_search_results_json', id='2f2b97c2-d67e-4d54-b3d5-83b5e5568602', tool_call_id='m4rmqxbv8', artifact={'query': 'electric vehicles market statistics 2026', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://tridenstechnology.com/electric-car-sales-statistics/', 'title': 'Global Electric Car Sales and Electric Vehicle Statistics (2026)', 'content': '| Electric Car Model | Sales |\n --- |\n| Tesla Model Y | 357,582 |\n| Tesla Model 3 | 192,440 |\n| Chevy Equinox EV | 57,945 |\n| Ford Mustang Mach-E | 51,620 |\n| Hyundai Ioniq 5 | 47,039 |\n| Honda Prologue | 41,194 |\n| Ford F-150 Lightning | 27,307 |\n| Rivian R1S | 24,852 |\n| Chevy Blazer EV | 22,637 |\n| Volkswagen ID.4 | 22,373 |\n\nConclusion\n\nBased on this research, we expect that the percentage of electric vehicles on our roads will increase to approximately 25% by the end of 2026.\n\nLoading ... Loading …\n\nIn conclusion, it seems that the future is looking bright for electric cars and 2026 is expected to be strong in EV sales.\n\nFor sure, we have just started the EV revolution!\n\nDo you agree? Drop the comment below. 😊\n\n## Ready to get started? [...] | Electric Car Model | Sales |\n --- |\n| Tesla Model Y | 8,360 |\n| Tesla Model 3 | 3,335 |\n| Volvo EX40 | 1,778 |\n| Volvo EX30 | 1,683 |\n| VW ID.7 | 1,599 |\n| VW ID.4 | 1,594 |\n| Skoda Elroq | 1,219 |\n| Ford Explorer | 1,135 |\n| Skoda Enyaq | 942 |\n| Nissan Ariya | 897 |\n\n### France Electric Cars Sales\n\nIn Q1 2025, electric vehicle sales in France fell by 7% compared to the same period in 2024, however the full year sales were higher in 2025.\n\nRenault 5 was the best-selling model in Q4 2025, with 12,374 new registrations.\n\nThese are bestsellers in France: [...] In 2025, electric car sales continued to grow: ~473,348 fully electric cars were registered, representing around 23.4 % of all new cars sold across the year and marking a ~21.6 % increase over 2024.\n\nThese were UK’s best-selling manufacturers in 2025:\n\n| Brand | Sales |\n --- |\n| Ford | 8.5% |\n| Tesla | 8.0% |\n| BYD | 7.0% |\n| VW | 6.6% |\n| AUDI | 6.2% |\n| Renault | 5.0% |\n| BMW | 4.8% |\n| Mercedes | 4.8% |\n| Skoda | 4.1% |\n| Hyundai | 4.0% |\n\nSource: CleanTechnica\n\n## Electric Car Sales in the US\n\nElectric car adoption in the US is happening at a faster pace than predicted.\n\nThe US charging infrastructure is getting built out fast, and Tesla’s opening the Supercharger network.\n\nElectric car sales in the USA reached new highs again in Q1 2025, continuing the long-term growth trend.', 'score': 0.95838624, 'raw_content': None}, {'url': 'https://ev-lectron.com/blogs/blog/electric-vehicle-forecast-2025-global-growth-policy-shifts-and-industry-outlook-leading-to-2026?srsltid=AfmBOorhtoH-Q7WMM8VwBmzv1P-rKQjMyoa-8aa1u_lNEk7G9Hej4aSt', 'title': 'Electric Vehicle Forecast 2026: Global Growth, Policy Shifts, and Indu', 'content': '## Global EV Market Outlook for 2026\n\nThe global EV market continues growing rapidly, but at a more measured pace than earlier years. Global sales of electric vehicles, including battery electric vehicles and plug-in hybrids, are expected to rise again in 2026 as electrification expands across passenger cars and light commercial vehicles.\n\nIn 2025, approximately 20.7 million EV units are expected to be registered globally, marking a 20% year-over-year growth. S&P Global Mobility projects global sales for battery electric passenger vehicles to post 15.1 million units for 2025, up by 30% compared to 2024 levels.\n\nA white Volvo EX30 electric SUV charging at an RACV public ultra-rapid charging station using a tethered CCS2 connector.', 'score': 0.9338243, 'raw_content': None}], 'response_time': 1.39, 'request_id': '523f320d-c386-470a-8c3d-715a32aa1e74'})]
last_message content='[{"title": "Global Electric Car Sales and Electric Vehicle Statistics (2026)", "url": "https://tridenstechnology.com/electric-car-sales-statistics/", "content": "| Electric Car Model | Sales |\\n --- |\\n| Tesla Model Y | 357,582 |\\n| Tesla Model 3 | 192,440 |\\n| Chevy Equinox EV | 57,945 |\\n| Ford Mustang Mach-E | 51,620 |\\n| Hyundai Ioniq 5 | 47,039 |\\n| Honda Prologue | 41,194 |\\n| Ford F-150 Lightning | 27,307 |\\n| Rivian R1S | 24,852 |\\n| Chevy Blazer EV | 22,637 |\\n| Volkswagen ID.4 | 22,373 |\\n\\nConclusion\\n\\nBased on this research, we expect that the percentage of electric vehicles on our roads will increase to approximately 25% by the end of 2026.\\n\\nLoading ... Loading …\\n\\nIn conclusion, it seems that the future is looking bright for electric cars and 2026 is expected to be strong in EV sales.\\n\\nFor sure, we have just started the EV revolution!\\n\\nDo you agree? Drop the comment below. 😊\\n\\n## Ready to get started? [...] | Electric Car Model | Sales |\\n --- |\\n| Tesla Model Y | 8,360 |\\n| Tesla Model 3 | 3,335 |\\n| Volvo EX40 | 1,778 |\\n| Volvo EX30 | 1,683 |\\n| VW ID.7 | 1,599 |\\n| VW ID.4 | 1,594 |\\n| Skoda Elroq | 1,219 |\\n| Ford Explorer | 1,135 |\\n| Skoda Enyaq | 942 |\\n| Nissan Ariya | 897 |\\n\\n### France Electric Cars Sales\\n\\nIn Q1 2025, electric vehicle sales in France fell by 7% compared to the same period in 2024, however the full year sales were higher in 2025.\\n\\nRenault 5 was the best-selling model in Q4 2025, with 12,374 new registrations.\\n\\nThese are bestsellers in France: [...] In 2025, electric car sales continued to grow: ~473,348 fully electric cars were registered, representing around 23.4 % of all new cars sold across the year and marking a ~21.6 % increase over 2024.\\n\\nThese were UK’s best-selling manufacturers in 2025:\\n\\n| Brand | Sales |\\n --- |\\n| Ford | 8.5% |\\n| Tesla | 8.0% |\\n| BYD | 7.0% |\\n| VW | 6.6% |\\n| AUDI | 6.2% |\\n| Renault | 5.0% |\\n| BMW | 4.8% |\\n| Mercedes | 4.8% |\\n| Skoda | 4.1% |\\n| Hyundai | 4.0% |\\n\\nSource: CleanTechnica\\n\\n## Electric Car Sales in the US\\n\\nElectric car adoption in the US is happening at a faster pace than predicted.\\n\\nThe US charging infrastructure is getting built out fast, and Tesla’s opening the Supercharger network.\\n\\nElectric car sales in the USA reached new highs again in Q1 2025, continuing the long-term growth trend.", "score": 0.95838624}, {"title": "Electric Vehicle Forecast 2026: Global Growth, Policy Shifts, and Indu", "url": "https://ev-lectron.com/blogs/blog/electric-vehicle-forecast-2025-global-growth-policy-shifts-and-industry-outlook-leading-to-2026?srsltid=AfmBOorhtoH-Q7WMM8VwBmzv1P-rKQjMyoa-8aa1u_lNEk7G9Hej4aSt", "content": "## Global EV Market Outlook for 2026\\n\\nThe global EV market continues growing rapidly, but at a more measured pace than earlier years. Global sales of electric vehicles, including battery electric vehicles and plug-in hybrids, are expected to rise again in 2026 as electrification expands across passenger cars and light commercial vehicles.\\n\\nIn 2025, approximately 20.7 million EV units are expected to be registered globally, marking a 20% year-over-year growth. S&P Global Mobility projects global sales for battery electric passenger vehicles to post 15.1 million units for 2025, up by 30% compared to 2024 levels.\\n\\nA white Volvo EX30 electric SUV charging at an RACV public ultra-rapid charging station using a tethered CCS2 connector.", "score": 0.9338243}]' name='tavily_search_results_json' id='2f2b97c2-d67e-4d54-b3d5-83b5e5568602' tool_call_id='m4rmqxbv8' artifact={'query': 'electric vehicles market statistics 2026', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://tridenstechnology.com/electric-car-sales-statistics/', 'title': 'Global Electric Car Sales and Electric Vehicle Statistics (2026)', 'content': '| Electric Car Model | Sales |\n --- |\n| Tesla Model Y | 357,582 |\n| Tesla Model 3 | 192,440 |\n| Chevy Equinox EV | 57,945 |\n| Ford Mustang Mach-E | 51,620 |\n| Hyundai Ioniq 5 | 47,039 |\n| Honda Prologue | 41,194 |\n| Ford F-150 Lightning | 27,307 |\n| Rivian R1S | 24,852 |\n| Chevy Blazer EV | 22,637 |\n| Volkswagen ID.4 | 22,373 |\n\nConclusion\n\nBased on this research, we expect that the percentage of electric vehicles on our roads will increase to approximately 25% by the end of 2026.\n\nLoading ... Loading …\n\nIn conclusion, it seems that the future is looking bright for electric cars and 2026 is expected to be strong in EV sales.\n\nFor sure, we have just started the EV revolution!\n\nDo you agree? Drop the comment below. 😊\n\n## Ready to get started? [...] | Electric Car Model | Sales |\n --- |\n| Tesla Model Y | 8,360 |\n| Tesla Model 3 | 3,335 |\n| Volvo EX40 | 1,778 |\n| Volvo EX30 | 1,683 |\n| VW ID.7 | 1,599 |\n| VW ID.4 | 1,594 |\n| Skoda Elroq | 1,219 |\n| Ford Explorer | 1,135 |\n| Skoda Enyaq | 942 |\n| Nissan Ariya | 897 |\n\n### France Electric Cars Sales\n\nIn Q1 2025, electric vehicle sales in France fell by 7% compared to the same period in 2024, however the full year sales were higher in 2025.\n\nRenault 5 was the best-selling model in Q4 2025, with 12,374 new registrations.\n\nThese are bestsellers in France: [...] In 2025, electric car sales continued to grow: ~473,348 fully electric cars were registered, representing around 23.4 % of all new cars sold across the year and marking a ~21.6 % increase over 2024.\n\nThese were UK’s best-selling manufacturers in 2025:\n\n| Brand | Sales |\n --- |\n| Ford | 8.5% |\n| Tesla | 8.0% |\n| BYD | 7.0% |\n| VW | 6.6% |\n| AUDI | 6.2% |\n| Renault | 5.0% |\n| BMW | 4.8% |\n| Mercedes | 4.8% |\n| Skoda | 4.1% |\n| Hyundai | 4.0% |\n\nSource: CleanTechnica\n\n## Electric Car Sales in the US\n\nElectric car adoption in the US is happening at a faster pace than predicted.\n\nThe US charging infrastructure is getting built out fast, and Tesla’s opening the Supercharger network.\n\nElectric car sales in the USA reached new highs again in Q1 2025, continuing the long-term growth trend.', 'score': 0.95838624, 'raw_content': None}, {'url': 'https://ev-lectron.com/blogs/blog/electric-vehicle-forecast-2025-global-growth-policy-shifts-and-industry-outlook-leading-to-2026?srsltid=AfmBOorhtoH-Q7WMM8VwBmzv1P-rKQjMyoa-8aa1u_lNEk7G9Hej4aSt', 'title': 'Electric Vehicle Forecast 2026: Global Growth, Policy Shifts, and Indu', 'content': '## Global EV Market Outlook for 2026\n\nThe global EV market continues growing rapidly, but at a more measured pace than earlier years. Global sales of electric vehicles, including battery electric vehicles and plug-in hybrids, are expected to rise again in 2026 as electrification expands across passenger cars and light commercial vehicles.\n\nIn 2025, approximately 20.7 million EV units are expected to be registered globally, marking a 20% year-over-year growth. S&P Global Mobility projects global sales for battery electric passenger vehicles to post 15.1 million units for 2025, up by 30% compared to 2024 levels.\n\nA white Volvo EX30 electric SUV charging at an RACV public ultra-rapid charging station using a tethered CCS2 connector.', 'score': 0.9338243, 'raw_content': None}], 'response_time': 1.39, 'request_id': '523f320d-c386-470a-8c3d-715a32aa1e74'}
resuming_from_tool True
messages [SystemMessage(content='\nYou are a professional presentation designer.\n\nYour job is to generate visually balanced slide content for a PowerPoint presentation.\n\nFor each slide choose the most appropriate layout or a mix of layouts  and generate structured content.\n\nAvailable layouts:\n- bullets\n- bullets_with_text\n- paragraph\n- mixed\n\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"slide_number": {"title": "Slide Number", "type": "integer"}, "slide_title": {"title": "Slide Title", "type": "string"}, "layout": {"enum": ["bullets", "bullets_with_text", "paragraph", "two_column", "mixed"], "title": "Layout", "type": "string"}, "intro_line": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Intro Line"}, "bullet_points": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Bullet Points"}, "supporting_text": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Supporting Text"}, "paragraphs": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Paragraphs"}}, "required": ["slide_number", "slide_title", "layout"]}\n```\n\n\nRules:\n- Slides should be informative but not crowded.\n- Include at least 1 statistic with source\n- Use clear, short content (120-150 words)\n- Bullet points: 5–7 items, short phrases\n- Keep slide visually balanced\nReturn JSON only.\n', additional_kwargs={}, response_metadata={}, id='535ddf52-7c91-43c7-b6c3-ee262b5b1b24'), HumanMessage(content='\nSlide: Introduction to Electric Vehicles\nTopic: Electric Vehicles and the Future of Transportation\n\nIf this slide needs current statistics, market data, or recent research, use the search tool first and get data till current 2026 year.\nFor conceptual, instructional, or introductory slides, generate content directly without searching.\n', additional_kwargs={}, response_metadata={}, id='e7d13a49-8a0f-4773-a726-673e057a681b'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'm4rmqxbv8', 'function': {'arguments': '{"query":"electric vehicles market statistics 2026"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 814, 'total_tokens': 838, 'completion_time': 0.069229028, 'completion_tokens_details': None, 'prompt_time': 0.042044197, 'prompt_tokens_details': None, 'queue_time': 0.048020733, 'total_time': 0.111273225}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d30eb-0be9-73e0-b0e9-dd1ffdcd9196-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'electric vehicles market statistics 2026'}, 'id': 'm4rmqxbv8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 814, 'output_tokens': 24, 'total_tokens': 838}), ToolMessage(content='[{"title": "Global Electric Car Sales and Electric Vehicle Statistics (2026)", "url": "https://tridenstechnology.com/electric-car-sales-statistics/", "content": "| Electric Car Model | Sales |\\n --- |\\n| Tesla Model Y | 357,582 |\\n| Tesla Model 3 | 192,440 |\\n| Chevy Equinox EV | 57,945 |\\n| Ford Mustang Mach-E | 51,620 |\\n| Hyundai Ioniq 5 | 47,039 |\\n| Honda Prologue | 41,194 |\\n| Ford F-150 Lightning | 27,307 |\\n| Rivian R1S | 24,852 |\\n| Chevy Blazer EV | 22,637 |\\n| Volkswagen ID.4 | 22,373 |\\n\\nConclusion\\n\\nBased on this research, we expect that the percentage of electric vehicles on our roads will increase to approximately 25% by the end of 2026.\\n\\nLoading ... Loading …\\n\\nIn conclusion, it seems that the future is looking bright for electric cars and 2026 is expected to be strong in EV sales.\\n\\nFor sure, we have just started the EV revolution!\\n\\nDo you agree? Drop the comment below. 😊\\n\\n## Ready to get started? [...] | Electric Car Model | Sales |\\n --- |\\n| Tesla Model Y | 8,360 |\\n| Tesla Model 3 | 3,335 |\\n| Volvo EX40 | 1,778 |\\n| Volvo EX30 | 1,683 |\\n| VW ID.7 | 1,599 |\\n| VW ID.4 | 1,594 |\\n| Skoda Elroq | 1,219 |\\n| Ford Explorer | 1,135 |\\n| Skoda Enyaq | 942 |\\n| Nissan Ariya | 897 |\\n\\n### France Electric Cars Sales\\n\\nIn Q1 2025, electric vehicle sales in France fell by 7% compared to the same period in 2024, however the full year sales were higher in 2025.\\n\\nRenault 5 was the best-selling model in Q4 2025, with 12,374 new registrations.\\n\\nThese are bestsellers in France: [...] In 2025, electric car sales continued to grow: ~473,348 fully electric cars were registered, representing around 23.4 % of all new cars sold across the year and marking a ~21.6 % increase over 2024.\\n\\nThese were UK’s best-selling manufacturers in 2025:\\n\\n| Brand | Sales |\\n --- |\\n| Ford | 8.5% |\\n| Tesla | 8.0% |\\n| BYD | 7.0% |\\n| VW | 6.6% |\\n| AUDI | 6.2% |\\n| Renault | 5.0% |\\n| BMW | 4.8% |\\n| Mercedes | 4.8% |\\n| Skoda | 4.1% |\\n| Hyundai | 4.0% |\\n\\nSource: CleanTechnica\\n\\n## Electric Car Sales in the US\\n\\nElectric car adoption in the US is happening at a faster pace than predicted.\\n\\nThe US charging infrastructure is getting built out fast, and Tesla’s opening the Supercharger network.\\n\\nElectric car sales in the USA reached new highs again in Q1 2025, continuing the long-term growth trend.", "score": 0.95838624}, {"title": "Electric Vehicle Forecast 2026: Global Growth, Policy Shifts, and Indu", "url": "https://ev-lectron.com/blogs/blog/electric-vehicle-forecast-2025-global-growth-policy-shifts-and-industry-outlook-leading-to-2026?srsltid=AfmBOorhtoH-Q7WMM8VwBmzv1P-rKQjMyoa-8aa1u_lNEk7G9Hej4aSt", "content": "## Global EV Market Outlook for 2026\\n\\nThe global EV market continues growing rapidly, but at a more measured pace than earlier years. Global sales of electric vehicles, including battery electric vehicles and plug-in hybrids, are expected to rise again in 2026 as electrification expands across passenger cars and light commercial vehicles.\\n\\nIn 2025, approximately 20.7 million EV units are expected to be registered globally, marking a 20% year-over-year growth. S&P Global Mobility projects global sales for battery electric passenger vehicles to post 15.1 million units for 2025, up by 30% compared to 2024 levels.\\n\\nA white Volvo EX30 electric SUV charging at an RACV public ultra-rapid charging station using a tethered CCS2 connector.", "score": 0.9338243}]', name='tavily_search_results_json', id='2f2b97c2-d67e-4d54-b3d5-83b5e5568602', tool_call_id='m4rmqxbv8', artifact={'query': 'electric vehicles market statistics 2026', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://tridenstechnology.com/electric-car-sales-statistics/', 'title': 'Global Electric Car Sales and Electric Vehicle Statistics (2026)', 'content': '| Electric Car Model | Sales |\n --- |\n| Tesla Model Y | 357,582 |\n| Tesla Model 3 | 192,440 |\n| Chevy Equinox EV | 57,945 |\n| Ford Mustang Mach-E | 51,620 |\n| Hyundai Ioniq 5 | 47,039 |\n| Honda Prologue | 41,194 |\n| Ford F-150 Lightning | 27,307 |\n| Rivian R1S | 24,852 |\n| Chevy Blazer EV | 22,637 |\n| Volkswagen ID.4 | 22,373 |\n\nConclusion\n\nBased on this research, we expect that the percentage of electric vehicles on our roads will increase to approximately 25% by the end of 2026.\n\nLoading ... Loading …\n\nIn conclusion, it seems that the future is looking bright for electric cars and 2026 is expected to be strong in EV sales.\n\nFor sure, we have just started the EV revolution!\n\nDo you agree? Drop the comment below. 😊\n\n## Ready to get started? [...] | Electric Car Model | Sales |\n --- |\n| Tesla Model Y | 8,360 |\n| Tesla Model 3 | 3,335 |\n| Volvo EX40 | 1,778 |\n| Volvo EX30 | 1,683 |\n| VW ID.7 | 1,599 |\n| VW ID.4 | 1,594 |\n| Skoda Elroq | 1,219 |\n| Ford Explorer | 1,135 |\n| Skoda Enyaq | 942 |\n| Nissan Ariya | 897 |\n\n### France Electric Cars Sales\n\nIn Q1 2025, electric vehicle sales in France fell by 7% compared to the same period in 2024, however the full year sales were higher in 2025.\n\nRenault 5 was the best-selling model in Q4 2025, with 12,374 new registrations.\n\nThese are bestsellers in France: [...] In 2025, electric car sales continued to grow: ~473,348 fully electric cars were registered, representing around 23.4 % of all new cars sold across the year and marking a ~21.6 % increase over 2024.\n\nThese were UK’s best-selling manufacturers in 2025:\n\n| Brand | Sales |\n --- |\n| Ford | 8.5% |\n| Tesla | 8.0% |\n| BYD | 7.0% |\n| VW | 6.6% |\n| AUDI | 6.2% |\n| Renault | 5.0% |\n| BMW | 4.8% |\n| Mercedes | 4.8% |\n| Skoda | 4.1% |\n| Hyundai | 4.0% |\n\nSource: CleanTechnica\n\n## Electric Car Sales in the US\n\nElectric car adoption in the US is happening at a faster pace than predicted.\n\nThe US charging infrastructure is getting built out fast, and Tesla’s opening the Supercharger network.\n\nElectric car sales in the USA reached new highs again in Q1 2025, continuing the long-term growth trend.', 'score': 0.95838624, 'raw_content': None}, {'url': 'https://ev-lectron.com/blogs/blog/electric-vehicle-forecast-2025-global-growth-policy-shifts-and-industry-outlook-leading-to-2026?srsltid=AfmBOorhtoH-Q7WMM8VwBmzv1P-rKQjMyoa-8aa1u_lNEk7G9Hej4aSt', 'title': 'Electric Vehicle Forecast 2026: Global Growth, Policy Shifts, and Indu', 'content': '## Global EV Market Outlook for 2026\n\nThe global EV market continues growing rapidly, but at a more measured pace than earlier years. Global sales of electric vehicles, including battery electric vehicles and plug-in hybrids, are expected to rise again in 2026 as electrification expands across passenger cars and light commercial vehicles.\n\nIn 2025, approximately 20.7 million EV units are expected to be registered globally, marking a 20% year-over-year growth. S&P Global Mobility projects global sales for battery electric passenger vehicles to post 15.1 million units for 2025, up by 30% compared to 2024 levels.\n\nA white Volvo EX30 electric SUV charging at an RACV public ultra-rapid charging station using a tethered CCS2 connector.', 'score': 0.9338243, 'raw_content': None}], 'response_time': 1.39, 'request_id': '523f320d-c386-470a-8c3d-715a32aa1e74'})]
output['messages'] [SystemMessage(content='\nYou are a professional presentation designer.\n\nYour job is to generate visually balanced slide content for a PowerPoint presentation.\n\nFor each slide choose the most appropriate layout or a mix of layouts  and generate structured content.\n\nAvailable layouts:\n- bullets\n- bullets_with_text\n- paragraph\n- mixed\n\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"slide_number": {"title": "Slide Number", "type": "integer"}, "slide_title": {"title": "Slide Title", "type": "string"}, "layout": {"enum": ["bullets", "bullets_with_text", "paragraph", "two_column", "mixed"], "title": "Layout", "type": "string"}, "intro_line": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Intro Line"}, "bullet_points": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Bullet Points"}, "supporting_text": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Supporting Text"}, "paragraphs": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Paragraphs"}}, "required": ["slide_number", "slide_title", "layout"]}\n```\n\n\nRules:\n- Slides should be informative but not crowded.\n- Include at least 1 statistic with source\n- Use clear, short content (120-150 words)\n- Bullet points: 5–7 items, short phrases\n- Keep slide visually balanced\nReturn JSON only.\n', additional_kwargs={}, response_metadata={}, id='535ddf52-7c91-43c7-b6c3-ee262b5b1b24'), HumanMessage(content='\nSlide: Introduction to Electric Vehicles\nTopic: Electric Vehicles and the Future of Transportation\n\nIf this slide needs current statistics, market data, or recent research, use the search tool first and get data till current 2026 year.\nFor conceptual, instructional, or introductory slides, generate content directly without searching.\n', additional_kwargs={}, response_metadata={})]
after append [SystemMessage(content='\nYou are a professional

In [13]:
l = """[HumanMessage(content='\nCreate EXACTLY 5 slide titles for a presentation on:\n\nTopic: Electric Vehicles and the Future of Transportation\n\nDo not create fewer or more slides.\n', additional_kwargs={}, response_metadata={}, id='79f9f33a-dab2-4c9e-b78f-6ef6b4dd20f5'), 
AIMessage(content='["Introduction to Electric Vehicles", "The Benefits of Electric Vehicles", "Challenges and Limitations of Electric Vehicles", "The Future of Transportation: Emerging Trends and Technologies", "Conclusion: Embracing a Sustainable Transportation Future"]', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 46, 'prompt_tokens': 399, 'total_tokens': 445, 'completion_time': 0.154857142, 'completion_tokens_details': None, 'prompt_time': 0.046790351, 'prompt_tokens_details': None, 'queue_time': 0.161466158, 'total_time': 0.201647493}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d30ea-fe48-7403-939f-79f9b26d694c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 399, 'output_tokens': 46, 'total_tokens': 445}), 
SystemMessage(content='\nYou are a professional presentation designer.\n\nYour job is to generate visually balanced slide content for a PowerPoint presentation.\n\nFor each slide choose the most appropriate layout or a mix of layouts  and generate structured content.\n\nAvailable layouts:\n- bullets\n- bullets_with_text\n- paragraph\n- mixed\n\n\nThe output should be formatted as a JSON instance that conforms to the JSON schema below.\n\nAs an example, for the schema {"properties": {"foo": {"title": "Foo", "description": "a list of strings", "type": "array", "items": {"type": "string"}}}, "required": ["foo"]}\nthe object {"foo": ["bar", "baz"]} is a well-formatted instance of the schema. The object {"properties": {"foo": ["bar", "baz"]}} is not well-formatted.\n\nHere is the output schema:\n```\n{"properties": {"slide_number": {"title": "Slide Number", "type": "integer"}, "slide_title": {"title": "Slide Title", "type": "string"}, "layout": {"enum": ["bullets", "bullets_with_text", "paragraph", "two_column", "mixed"], "title": "Layout", "type": "string"}, "intro_line": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Intro Line"}, "bullet_points": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Bullet Points"}, "supporting_text": {"anyOf": [{"type": "string"}, {"type": "null"}], "default": null, "title": "Supporting Text"}, "paragraphs": {"anyOf": [{"items": {"type": "string"}, "type": "array"}, {"type": "null"}], "default": null, "title": "Paragraphs"}}, "required": ["slide_number", "slide_title", "layout"]}\n```\n\n\nRules:\n- Slides should be informative but not crowded.\n- Include at least 1 statistic with source\n- Use clear, short content (120-150 words)\n- Bullet points: 5–7 items, short phrases\n- Keep slide visually balanced\nReturn JSON only.\n', additional_kwargs={}, response_metadata={}, id='535ddf52-7c91-43c7-b6c3-ee262b5b1b24'), HumanMessage(content='\nSlide: Introduction to Electric Vehicles\nTopic: Electric Vehicles and the Future of Transportation\n\nIf this slide needs current statistics, market data, or recent research, use the search tool first and get data till current 2026 year.\nFor conceptual, instructional, or introductory slides, generate content directly without searching.\n', additional_kwargs={}, response_metadata={}, id='e7d13a49-8a0f-4773-a726-673e057a681b'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'm4rmqxbv8', 'function': {'arguments': '{"query":"electric vehicles market statistics 2026"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 814, 'total_tokens': 838, 'completion_time': 0.069229028, 'completion_tokens_details': None, 'prompt_time': 0.042044197, 'prompt_tokens_details': None, 'queue_time': 0.048020733, 'total_time': 0.111273225}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d30eb-0be9-73e0-b0e9-dd1ffdcd9196-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'electric vehicles market statistics 2026'}, 'id': 'm4rmqxbv8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 814, 'output_tokens': 24, 'total_tokens': 838}), ToolMessage(content='[{"title": "Global Electric Car Sales and Electric Vehicle Statistics (2026)", "url": "https://tridenstechnology.com/electric-car-sales-statistics/", "content": "| Electric Car Model | Sales |\\n --- |\\n| Tesla Model Y | 357,582 |\\n| Tesla Model 3 | 192,440 |\\n| Chevy Equinox EV | 57,945 |\\n| Ford Mustang Mach-E | 51,620 |\\n| Hyundai Ioniq 5 | 47,039 |\\n| Honda Prologue | 41,194 |\\n| Ford F-150 Lightning | 27,307 |\\n| Rivian R1S | 24,852 |\\n| Chevy Blazer EV | 22,637 |\\n| Volkswagen ID.4 | 22,373 |\\n\\nConclusion\\n\\nBased on this research, we expect that the percentage of electric vehicles on our roads will increase to approximately 25% by the end of 2026.\\n\\nLoading ... Loading …\\n\\nIn conclusion, it seems that the future is looking bright for electric cars and 2026 is expected to be strong in EV sales.\\n\\nFor sure, we have just started the EV revolution!\\n\\nDo you agree? Drop the comment below. 😊\\n\\n## Ready to get started? [...] | Electric Car Model | Sales |\\n --- |\\n| Tesla Model Y | 8,360 |\\n| Tesla Model 3 | 3,335 |\\n| Volvo EX40 | 1,778 |\\n| Volvo EX30 | 1,683 |\\n| VW ID.7 | 1,599 |\\n| VW ID.4 | 1,594 |\\n| Skoda Elroq | 1,219 |\\n| Ford Explorer | 1,135 |\\n| Skoda Enyaq | 942 |\\n| Nissan Ariya | 897 |\\n\\n### France Electric Cars Sales\\n\\nIn Q1 2025, electric vehicle sales in France fell by 7% compared to the same period in 2024, however the full year sales were higher in 2025.\\n\\nRenault 5 was the best-selling model in Q4 2025, with 12,374 new registrations.\\n\\nThese are bestsellers in France: [...] In 2025, electric car sales continued to grow: ~473,348 fully electric cars were registered, representing around 23.4 % of all new cars sold across the year and marking a ~21.6 % increase over 2024.\\n\\nThese were UK’s best-selling manufacturers in 2025:\\n\\n| Brand | Sales |\\n --- |\\n| Ford | 8.5% |\\n| Tesla | 8.0% |\\n| BYD | 7.0% |\\n| VW | 6.6% |\\n| AUDI | 6.2% |\\n| Renault | 5.0% |\\n| BMW | 4.8% |\\n| Mercedes | 4.8% |\\n| Skoda | 4.1% |\\n| Hyundai | 4.0% |\\n\\nSource: CleanTechnica\\n\\n## Electric Car Sales in the US\\n\\nElectric car adoption in the US is happening at a faster pace than predicted.\\n\\nThe US charging infrastructure is getting built out fast, and Tesla’s opening the Supercharger network.\\n\\nElectric car sales in the USA reached new highs again in Q1 2025, continuing the long-term growth trend.", "score": 0.95838624}, {"title": "Electric Vehicle Forecast 2026: Global Growth, Policy Shifts, and Indu", "url": "https://ev-lectron.com/blogs/blog/electric-vehicle-forecast-2025-global-growth-policy-shifts-and-industry-outlook-leading-to-2026?srsltid=AfmBOorhtoH-Q7WMM8VwBmzv1P-rKQjMyoa-8aa1u_lNEk7G9Hej4aSt", "content": "## Global EV Market Outlook for 2026\\n\\nThe global EV market continues growing rapidly, but at a more measured pace than earlier years. Global sales of electric vehicles, including battery electric vehicles and plug-in hybrids, are expected to rise again in 2026 as electrification expands across passenger cars and light commercial vehicles.\\n\\nIn 2025, approximately 20.7 million EV units are expected to be registered globally, marking a 20% year-over-year growth. S&P Global Mobility projects global sales for battery electric passenger vehicles to post 15.1 million units for 2025, up by 30% compared to 2024 levels.\\n\\nA white Volvo EX30 electric SUV charging at an RACV public ultra-rapid charging station using a tethered CCS2 connector.", "score": 0.9338243}]', name='tavily_search_results_json', id='2f2b97c2-d67e-4d54-b3d5-83b5e5568602', tool_call_id='m4rmqxbv8', artifact={'query': 'electric vehicles market statistics 2026', 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://tridenstechnology.com/electric-car-sales-statistics/', 'title': 'Global Electric Car Sales and Electric Vehicle Statistics (2026)', 'content': '| Electric Car Model | Sales |\n --- |\n| Tesla Model Y | 357,582 |\n| Tesla Model 3 | 192,440 |\n| Chevy Equinox EV | 57,945 |\n| Ford Mustang Mach-E | 51,620 |\n| Hyundai Ioniq 5 | 47,039 |\n| Honda Prologue | 41,194 |\n| Ford F-150 Lightning | 27,307 |\n| Rivian R1S | 24,852 |\n| Chevy Blazer EV | 22,637 |\n| Volkswagen ID.4 | 22,373 |\n\nConclusion\n\nBased on this research, we expect that the percentage of electric vehicles on our roads will increase to approximately 25% by the end of 2026.\n\nLoading ... Loading …\n\nIn conclusion, it seems that the future is looking bright for electric cars and 2026 is expected to be strong in EV sales.\n\nFor sure, we have just started the EV revolution!\n\nDo you agree? Drop the comment below. 😊\n\n## Ready to get started? [...] | Electric Car Model | Sales |\n --- |\n| Tesla Model Y | 8,360 |\n| Tesla Model 3 | 3,335 |\n| Volvo EX40 | 1,778 |\n| Volvo EX30 | 1,683 |\n| VW ID.7 | 1,599 |\n| VW ID.4 | 1,594 |\n| Skoda Elroq | 1,219 |\n| Ford Explorer | 1,135 |\n| Skoda Enyaq | 942 |\n| Nissan Ariya | 897 |\n\n### France Electric Cars Sales\n\nIn Q1 2025, electric vehicle sales in France fell by 7% compared to the same period in 2024, however the full year sales were higher in 2025.\n\nRenault 5 was the best-selling model in Q4 2025, with 12,374 new registrations.\n\nThese are bestsellers in France: [...] In 2025, electric car sales continued to grow: ~473,348 fully electric cars were registered, representing around 23.4 % of all new cars sold across the year and marking a ~21.6 % increase over 2024.\n\nThese were UK’s best-selling manufacturers in 2025:\n\n| Brand | Sales |\n --- |\n| Ford | 8.5% |\n| Tesla | 8.0% |\n| BYD | 7.0% |\n| VW | 6.6% |\n| AUDI | 6.2% |\n| Renault | 5.0% |\n| BMW | 4.8% |\n| Mercedes | 4.8% |\n| Skoda | 4.1% |\n| Hyundai | 4.0% |\n\nSource: CleanTechnica\n\n## Electric Car Sales in the US\n\nElectric car adoption in the US is happening at a faster pace than predicted.\n\nThe US charging infrastructure is getting built out fast, and Tesla’s opening the Supercharger network.\n\nElectric car sales in the USA reached new highs again in Q1 2025, continuing the long-term growth trend.', 'score': 0.95838624, 'raw_content': None}, {'url': 'https://ev-lectron.com/blogs/blog/electric-vehicle-forecast-2025-global-growth-policy-shifts-and-industry-outlook-leading-to-2026?srsltid=AfmBOorhtoH-Q7WMM8VwBmzv1P-rKQjMyoa-8aa1u_lNEk7G9Hej4aSt', 'title': 'Electric Vehicle Forecast 2026: Global Growth, Policy Shifts, and Indu', 'content': '## Global EV Market Outlook for 2026\n\nThe global EV market continues growing rapidly, but at a more measured pace than earlier years. Global sales of electric vehicles, including battery electric vehicles and plug-in hybrids, are expected to rise again in 2026 as electrification expands across passenger cars and light commercial vehicles.\n\nIn 2025, approximately 20.7 million EV units are expected to be registered globally, marking a 20% year-over-year growth. S&P Global Mobility projects global sales for battery electric passenger vehicles to post 15.1 million units for 2025, up by 30% compared to 2024 levels.\n\nA white Volvo EX30 electric SUV charging at an RACV public ultra-rapid charging station using a tethered CCS2 connector.', 'score': 0.9338243, 'raw_content': None}], 'response_time': 1.39, 'request_id': '523f320d-c386-470a-8c3d-715a32aa1e74'}), HumanMessage(content='\nSlide: Introduction to Electric Vehicles\nTopic: Electric Vehicles and the Future of Transportation\n\nIf this slide needs current statistics, market data, or recent research, use the search tool first and get data till current 2026 year.\nFor conceptual, instructional, or introductory slides, generate content directly without searching.\n', additional_kwargs={}, response_metadata={}, id='f6c71494-13f5-44e5-be19-e0507be5bab5'), AIMessage(content='{"slide_number": 1, "slide_title": "Introduction to Electric Vehicles", "layout": "bullets_with_text", "intro_line": "Electric vehicles are transforming the future of transportation", "bullet_points": ["25% of vehicles on the road will be electric by 2026", "20.7 million EV units are expected to be registered globally in 2025", "15.1 million battery electric passenger vehicles are projected to be sold in 2025"], "supporting_text": "The global EV market continues to grow rapidly, with a expected 20% year-over-year growth in 2025. Source: CleanTechnica, S&P Global Mobility", "paragraphs": null}', additional_kwargs={}, response_metadata={'token_usage': {'completion_tokens': 145, 'prompt_tokens': 1885, 'total_tokens': 2030, 'completion_time': 0.395341921, 'completion_tokens_details': None, 'prompt_time': 0.167460497, 'prompt_tokens_details': None, 'queue_time': 0.160452151, 'total_time': 0.562802418}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_45180df409', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d30eb-180e-7502-9a68-d718833e3433-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 1885, 'output_tokens': 145, 'total_tokens': 2030}), HumanMessage(content='\nSlide: The Benefits of Electric Vehicles\nTopic: Electric Vehicles and the Future of Transportation\n\nIf this slide needs current statistics, market data, or recent research, use the search tool first and get data till current 2026 year.\nFor conceptual, instructional, or introductory slides, generate content directly without searching.\n', additional_kwargs={}, response_metadata={}, id='247cde45-0a95-4a7a-b274-c60a0e30817a'), AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'y6t3jk2x8', 'function': {'arguments': '{"query":"electric vehicles market statistics 2026"}', 'name': 'tavily_search_results_json'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 24, 'prompt_tokens': 815, 'total_tokens': 839, 'completion_time': 0.074625216, 'completion_tokens_details': None, 'prompt_time': 0.042265904, 'prompt_tokens_details': None, 'queue_time': 0.049400796, 'total_time': 0.11689112}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d30eb-708d-7552-8552-c92618d26ea3-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'electric vehicles market statistics 2026'}, 'id': 'y6t3jk2x8', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 815, 'output_tokens': 24, 'total_tokens': 839}), ToolMessage(content='[{"title": "Global Electric Car Sales and Electric Vehicle Statistics (2026)", "url": "https://tridenstechnology.com/electric-car-sales-statistics/", "content": "| Electric Car Model | Sales |\\n --- |\\n| Tesla Model Y | 357,582 |\\n| Tesla Model 3 | 192,440 |\\n| Chevy Equinox EV | 57,945 |\\n| Ford Mustang Mach-E | 51,620 |\\n| Hyundai Ioniq 5 | 47,039 |\\n| Honda Prologue | 41,194 |\\n| Ford F-150 Lightning | 27,307 |\\n| Rivian R1S | 24,852 |\\n| Chevy Blazer EV | 22,637 |\\n| Volkswagen ID.4 | 22,373 |\\n\\nConclusion\\n\\nBased on this research, we expect that the percentage of electric vehicles on our roads will increase to approximately 25% by the end of 2026.\\n\\nLoading ... Loading …\\n\\nIn conclusion, it seems that the future is looking bright for electric cars and 2026 is expected to be strong in EV sales.\\n\\nFor sure, we have just started the EV revolution!\\n\\nDo you agree? Drop the comment below. 😊\\n\\n## Ready to get started? [...] | Electric Car Model | Sales |\\n --- |\\n| Tesla Model Y | 8,360 |\\n| Tesla Model 3 | 3,335 |\\n| Volvo EX40 | 1,778 |\\n| Volvo EX30 | 1,683 |\\n| VW ID.7 | 1,599 |\\n| VW ID.4 | 1,594 |\\n| Skoda Elroq | 1,219 |\\n| Ford Explorer | 1,135 |\\n| Skoda Enyaq | 942 |\\n| Nissan Ariya | 897 |\\n\\n### France Electric Cars Sales\\n\\nIn Q1 2025, electric vehicle sales in France fell by 7% compared to the same period in 2024, however the full year sales were higher in 2025.\\n\\nRenault 5 was the best-selling model in Q4 2025, with 12,374 new registrations.\\n\\nThese are bestsellers in France: [...] In 2025, electric car sales continued to grow: ~473,348 fully electric cars were registered, representing around 23.4 % of all new cars sold across the year and marking a ~21.6 % increase over 2024.\\n\\nThese were UK’s best-selling manufacturers in 2025:\\n\\n| Brand | Sales |\\n --- |\\n| Ford | 8.5% |\\n| Tesla | 8.0% |\\n| BYD | 7.0% |\\n| VW | 6.6% |\\n| AUDI | 6.2% |\\n| Renault | 5.0% |\\n| BMW | 4.8% |\\n| Mercedes | 4.8% |\\n| Skoda | 4.1% |\\n| Hyundai | 4.0% |\\n\\nSource: CleanTechnica\\n\\n## Electric Car Sales in the US\\n\\nElectric car adoption in the US is happening at a faster pace than predicted.\\n\\nThe US charging infrastructure is getting built out fast, and Tesla’s opening the Supercharger network.\\n\\nElectric car sales in the USA reached new highs again in Q1 2025, continuing the long-term growth trend.", "score": 0.95838624}, {"title": "Electric Vehicle Forecast 2026: Global Growth, Policy Shifts, and Indu", "url": "https://ev-lectron.com/blogs/blog/electric-vehicle-forecast-2025-global-growth-policy-shifts-and-industry-outlook-leading-to-2026?srsltid=AfmBOorhtoH-Q7WMM8VwBmzv1P-rKQjMyoa-8aa1u_lNEk7G9Hej4aSt", "content": "## Global EV Market Outlook for 2026\\n\\nThe global EV market continues growing rapidly, but at a more measured pace than earlier years. Global sales of electric vehicles, including battery electric vehicles and plug-in hybrids, are expected to rise again in 2026 as electrification expands across passenger cars and light commercial vehicles.\\n\\nIn 2025, approximately 20.7 million EV units are expected to be registered globally, marking a 20% year-over-year growth. S&P Global Mobility projects global sales for battery electric passenger vehicles to post 15.1 million units for 2025, up by 30% compared to 2024 levels.\\n\\nA white Volvo EX30 electric SUV charging at an RACV public ultra-rapid charging station using a tethered CCS2 connector.", "score": 0.9338243}]', name='tavily_search_results_json', id='aae589f3-e703-41ca-bd1f-c6c24acb5adf', tool_call_id='y6t3jk2x8', artifact={'query': 'electric vehicles market statistics 2026', 'response_time': 1.39, 'follow_up_questions': None, 'answer': None, 'images': [], 'results': [{'url': 'https://tridenstechnology.com/electric-car-sales-statistics/', 'title': 'Global Electric Car Sales and Electric Vehicle Statistics (2026)', 'content': '| Electric Car Model | Sales |\n --- |\n| Tesla Model Y | 357,582 |\n| Tesla Model 3 | 192,440 |\n| Chevy Equinox EV | 57,945 |\n| Ford Mustang Mach-E | 51,620 |\n| Hyundai Ioniq 5 | 47,039 |\n| Honda Prologue | 41,194 |\n| Ford F-150 Lightning | 27,307 |\n| Rivian R1S | 24,852 |\n| Chevy Blazer EV | 22,637 |\n| Volkswagen ID.4 | 22,373 |\n\nConclusion\n\nBased on this research, we expect that the percentage of electric vehicles on our roads will increase to approximately 25% by the end of 2026.\n\nLoading ... Loading …\n\nIn conclusion, it seems that the future is looking bright for electric cars and 2026 is expected to be strong in EV sales.\n\nFor sure, we have just started the EV revolution!\n\nDo you agree? Drop the comment below. 😊\n\n## Ready to get started? [...] | Electric Car Model | Sales |\n --- |\n| Tesla Model Y | 8,360 |\n| Tesla Model 3 | 3,335 |\n| Volvo EX40 | 1,778 |\n| Volvo EX30 | 1,683 |\n| VW ID.7 | 1,599 |\n| VW ID.4 | 1,594 |\n| Skoda Elroq | 1,219 |\n| Ford Explorer | 1,135 |\n| Skoda Enyaq | 942 |\n| Nissan Ariya | 897 |\n\n### France Electric Cars Sales\n\nIn Q1 2025, electric vehicle sales in France fell by 7% compared to the same period in 2024, however the full year sales were higher in 2025.\n\nRenault 5 was the best-selling model in Q4 2025, with 12,374 new registrations.\n\nThese are bestsellers in France: [...] In 2025, electric car sales continued to grow: ~473,348 fully electric cars were registered, representing around 23.4 % of all new cars sold across the year and marking a ~21.6 % increase over 2024.\n\nThese were UK’s best-selling manufacturers in 2025:\n\n| Brand | Sales |\n --- |\n| Ford | 8.5% |\n| Tesla | 8.0% |\n| BYD | 7.0% |\n| VW | 6.6% |\n| AUDI | 6.2% |\n| Renault | 5.0% |\n| BMW | 4.8% |\n| Mercedes | 4.8% |\n| Skoda | 4.1% |\n| Hyundai | 4.0% |\n\nSource: CleanTechnica\n\n## Electric Car Sales in the US\n\nElectric car adoption in the US is happening at a faster pace than predicted.\n\nThe US charging infrastructure is getting built out fast, and Tesla’s opening the Supercharger network.\n\nElectric car sales in the USA reached new highs again in Q1 2025, continuing the long-term growth trend.', 'score': 0.95838624, 'raw_content': None}, {'url': 'https://ev-lectron.com/blogs/blog/electric-vehicle-forecast-2025-global-growth-policy-shifts-and-industry-outlook-leading-to-2026?srsltid=AfmBOorhtoH-Q7WMM8VwBmzv1P-rKQjMyoa-8aa1u_lNEk7G9Hej4aSt', 'title': 'Electric Vehicle Forecast 2026: Global Growth, Policy Shifts, and Indu', 'content': '## Global EV Market Outlook for 2026\n\nThe global EV market continues growing rapidly, but at a more measured pace than earlier years. Global sales of electric vehicles, including battery electric vehicles and plug-in hybrids, are expected to rise again in 2026 as electrification expands across passenger cars and light commercial vehicles.\n\nIn 2025, approximately 20.7 million EV units are expected to be registered globally, marking a 20% year-over-year growth. S&P Global Mobility projects global sales for battery electric passenger vehicles to post 15.1 million units for 2025, up by 30% compared to 2024 levels.\n\nA white Volvo EX30 electric SUV charging at an RACV public ultra-rapid charging station using a tethered CCS2 connector.', 'score': 0.9338243, 'raw_content': None}], 'request_id': '8ca23dab-ed30-4a85-99a5-275182d75188'})]
""".split('),')
l

["[HumanMessage(content='\nCreate EXACTLY 5 slide titles for a presentation on:\n\nTopic: Electric Vehicles and the Future of Transportation\n\nDo not create fewer or more slides.\n', additional_kwargs={}, response_metadata={}, id='79f9f33a-dab2-4c9e-b78f-6ef6b4dd20f5'",
 ' \nAIMessage(content=\'["Introduction to Electric Vehicles", "The Benefits of Electric Vehicles", "Challenges and Limitations of Electric Vehicles", "The Future of Transportation: Emerging Trends and Technologies", "Conclusion: Embracing a Sustainable Transportation Future"]\', additional_kwargs={}, response_metadata={\'token_usage\': {\'completion_tokens\': 46, \'prompt_tokens\': 399, \'total_tokens\': 445, \'completion_time\': 0.154857142, \'completion_tokens_details\': None, \'prompt_time\': 0.046790351, \'prompt_tokens_details\': None, \'queue_time\': 0.161466158, \'total_time\': 0.201647493}, \'model_name\': \'llama-3.3-70b-versatile\', \'system_fingerprint\': \'fp_45180df409\', \'service_tier\': \'on_demand\', 

In [ ]:
[1,3,4]+[1,2]